In [ ]:
%%capture
!pip install "langchain==0.2.7"
!pip install "langchain-core==0.2.20"
!pip install "langchain-text-splitters==0.2.2"
!pip install "lxml==5.2.2"

In [ ]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/YRYau14UJyh0DdiLDdzFcA/companypolicies.txt"

# This is a long document you can split up.
with open("companypolicies.txt") as f:
    companypolicies = f.read()

print(companypolicies)

--2026-11-12 00:42:38--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/YRYau14UJyh0DdiLDdzFcA/companypolicies.txt
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15660 (15K) [text/plain]
Saving to: ‘companypolicies.txt’

companypolicies.txt 100%[===================>]  15.29K  --.-KB/s    in 0s      

2026-11-12 00:42:38 (200 MB/s) - ‘companypolicies.txt’ saved [15660/15660]

1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to th

Here, you can use an example to guide you through creating a document object. Langchain uses this object type to deal with text/documents.


In [ ]:
from langchain_core.documents import Document
Document(page_content="""Python is an interpreted high-level general-purpose programming language.
                        Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",
         metadata={
             'my_document_id' : 234234,
             'my_document_source' : "About Python",
             'my_document_create_time' : 1680013019
         })

Document(metadata={'my_document_id': 234234, 'my_document_source': 'About Python', 'my_document_create_time': 1680013019}, page_content="Python is an interpreted high-level general-purpose programming language.\n                        Python's design philosophy emphasizes code readability with its notable use of significant indentation.")

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
)

texts = text_splitter.split_text(companypolicies)
texts
len(texts)

texts = text_splitter.create_documents([companypolicies], metadatas=[{"document":"Company Policies"}])  # pass the metadata as well

texts[0]

Document(metadata={'document': 'Company Policies'}, page_content='1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built')

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
)
texts = text_splitter.create_documents([companypolicies])
texts
len(texts)

215

In [ ]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
[e.value for e in Language]
RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON)

['\nclass ', '\ndef ', '\n\tdef ', '\n\n', '\n', ' ', '']

In [ ]:
PYTHON_CODE = """
    def hello_world():
        print("Hello, World!")

    # Call the function
    hello_world()
"""
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0
)
python_docs = python_splitter.create_documents([PYTHON_CODE])
python_docs

[Document(page_content='def hello_world():'),
 Document(page_content='print("Hello, World!")'),
 Document(page_content='# Call the function\n    hello_world()')]

In [ ]:
RecursiveCharacterTextSplitter.get_separators_for_language(Language.JS)
JS_CODE = """
    function helloWorld() {
      console.log("Hello, World!");
    }

    // Call the function
    helloWorld();
"""

js_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.JS, chunk_size=60, chunk_overlap=0
)
js_docs = js_splitter.create_documents([JS_CODE])
js_docs

[Document(page_content='function helloWorld() {'),
 Document(page_content='console.log("Hello, World!");\n    }'),
 Document(page_content='// Call the function\n    helloWorld();')]

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
md = "# Foo\n\n## Bar\n\nHi this is Jim\n\nHi this is Joe\n\n### Boo \n\nHi this is Lance \n\n## Baz\n\nHi this is Molly"
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(md)
md_header_splits
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)
md_header_splits = markdown_splitter.split_text(md)
md_header_splits

[Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar'}, page_content='# Foo  \n## Bar  \nHi this is Jim  \nHi this is Joe'),
 Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar', 'Header 3': 'Boo'}, page_content='### Boo  \nHi this is Lance'),
 Document(metadata={'Header 1': 'Foo', 'Header 2': 'Baz'}, page_content='## Baz  \nHi this is Molly')]

In [ ]:
from langchain_text_splitters import HTMLHeaderTextSplitter
html_string = """
    <!DOCTYPE html>
    <html>
    <body>
        <div>
            <h1>Foo</h1>
            <p>Some intro text about Foo.</p>
            <div>
                <h2>Bar main section</h2>
                <p>Some intro text about Bar.</p>
                <h3>Bar subsection 1</h3>
                <p>Some text about the first subtopic of Bar.</p>
                <h3>Bar subsection 2</h3>
                <p>Some text about the second subtopic of Bar.</p>
            </div>
            <div>
                <h2>Baz</h2>
                <p>Some text about Baz</p>
            </div>
            <br>
            <p>Some concluding text about Foo</p>
        </div>
    </body>
    </html>
"""
headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
html_header_splits = html_splitter.split_text(html_string)
html_header_splits

[Document(page_content='Foo'),
 Document(metadata={'Header 1': 'Foo'}, page_content='Some intro text about Foo.  \nBar main section Bar subsection 1 Bar subsection 2'),
 Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar main section'}, page_content='Some intro text about Bar.'),
 Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar main section', 'Header 3': 'Bar subsection 1'}, page_content='Some text about the first subtopic of Bar.'),
 Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar main section', 'Header 3': 'Bar subsection 2'}, page_content='Some text about the second subtopic of Bar.'),
 Document(metadata={'Header 1': 'Foo'}, page_content='Baz'),
 Document(metadata={'Header 1': 'Foo', 'Header 2': 'Baz'}, page_content='Some text about Baz'),
 Document(metadata={'Header 1': 'Foo'}, page_content='Some concluding text about Foo')]

In [ ]:
from langchain_text_splitters import HTMLSectionSplitter

html_string = """
    <!DOCTYPE html>
    <html>
    <body>
        <div>
            <h1>Foo</h1>
            <p>Some intro text about Foo.</p>
            <div>
                <h2>Bar main section</h2>
                <p>Some intro text about Bar.</p>
                <h3>Bar subsection 1</h3>
                <p>Some text about the first subtopic of Bar.</p>
                <h3>Bar subsection 2</h3>
                <p>Some text about the second subtopic of Bar.</p>
            </div>
            <div>
                <h2>Baz</h2>
                <p>Some text about Baz</p>
            </div>
            <br>
            <p>Some concluding text about Foo</p>
        </div>
    </body>
    </html>
"""

headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2"), ("h3", "Header 3")]

html_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)
html_header_splits = html_splitter.split_text(html_string)
html_header_splits

[Document(metadata={'Header 1': 'Foo'}, page_content='Foo \n Some intro text about Foo.'),
 Document(metadata={'Header 2': 'Bar main section'}, page_content='Bar main section \n Some intro text about Bar.'),
 Document(metadata={'Header 3': 'Bar subsection 1'}, page_content='Bar subsection 1 \n Some text about the first subtopic of Bar.'),
 Document(metadata={'Header 3': 'Bar subsection 2'}, page_content='Bar subsection 2 \n Some text about the second subtopic of Bar.'),
 Document(metadata={'Header 2': 'Baz'}, page_content='Baz \n Some text about Baz \n \n \n Some concluding text about Foo')]

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# Configure the splitter using "\n" as the separator
newline_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
)

# Split the document
newline_texts = newline_splitter.create_documents([companypolicies])

print(f"Total chunks generated: {len(newline_texts)}")
print("\nFirst chunk sample:")
print(newline_texts[0])

Total chunks generated: 74

First chunk sample:
page_content='1.	Code of Conduct'


In [ ]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

latex_text = """
    \documentclass{article}

    \begin{document}

    \maketitle

    \section{Introduction}
    Large language models (LLMs) are a type of machine learning model that can be trained on vast amounts of text data to generate human-like language. In recent years, LLMs have made significant advances in a variety of natural language processing tasks, including language translation, text generation, and sentiment analysis.

    \subsection{History of LLMs}
    The earliest LLMs were developed in the 1980s and 1990s, but they were limited by the amount of data that could be processed and the computational power available at the time. In the past decade, however, advances in hardware and software have made it possible to train LLMs on massive datasets, leading to significant improvements in performance.

    \subsection{Applications of LLMs}
    LLMs have many applications in industry, including chatbots, content creation, and virtual assistants. They can also be used in academia for research in linguistics, psychology, and computational linguistics.

    \end{document}
"""

# Configure the LaTeX text splitter
latex_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.LATEX, chunk_size=150, chunk_overlap=0
)

# Split the document
latex_docs = latex_splitter.create_documents([latex_text])
for i, doc in enumerate(latex_docs):
    print(f"Chunk {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Chunk 1:
\documentclass{article}
    
    egin{document}
    
    \maketitle
    
    \section{Introduction}
    Large language models (LLMs) are a type
--------------------------------------------------
Chunk 2:
of machine learning model that can be trained on vast amounts of text data to generate human-like language. In recent years, LLMs have made
--------------------------------------------------
Chunk 3:
significant advances in a variety of natural language processing tasks, including language translation, text generation, and sentiment analysis.
--------------------------------------------------
Chunk 4:
\subsection{History of LLMs}
    The earliest LLMs were developed in the 1980s and 1990s, but they were limited by the amount of data that could
--------------------------------------------------
Chunk 5:
be processed and the computational power available at the time. In the past decade, however, advances in hardware and software have made it possible
---------------------------

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_466/1289130023.py:4: SyntaxWarning: invalid escape sequence '\d'
  \documentclass{article}


Full Code Lesson 2 Section

In [19]:
import re

sentence = "The cat sat on the mat. It was a sunny day."

# Bad cut: blindly cut after 20 characters
bad_cut = [sentence[:20], sentence[20:]]
print("BAD :", bad_cut)

# Good cut: cut right after each full stop
good_cut = re.split(r"(?<=\.)\s", sentence)
print("GOOD:", good_cut)

# Output:
# BAD : ['The cat sat on the m', 'at. It was a sunny day.']
# GOOD: ['The cat sat on the mat.', 'It was a sunny day.']

BAD : ['The cat sat on the m', 'at. It was a sunny day.']
GOOD: ['The cat sat on the mat.', 'It was a sunny day.']


In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1) LOAD: here we simply use a string instead of a real file
document = (
    "Employees receive 20 days of paid annual leave per year.\n\n"
    "Sick leave is granted for up to 10 days per year.\n\n"
    "Passwords must be changed every 90 days."
)

# 2) SPLIT: the step this whole lesson is about
splitter = RecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=0)
chunks = splitter.split_text(document)

# 3) EMBED + 4) STORE: skipped, we keep chunks in a plain Python list

# 5) RETRIEVE: pick the chunk that shares the most words with the question
def retrieve(question, chunks):
    q_words = set(question.lower().split())
    return max(chunks, key=lambda c: len(q_words & set(c.lower().split())))

best_chunk = retrieve("How many days of sick leave?", chunks)

# 6) GENERATE: a real system would send best_chunk to an LLM here
print("Chunks:", chunks)
print("Best chunk for the question:", best_chunk)

# Output:
# Chunks: ['Employees receive 20 days of paid annual leave per year.',
#          'Sick leave is granted for up to 10 days per year.',
#          'Passwords must be changed every 90 days.']
# Best chunk for the question: Sick leave is granted for up to 10 days per year.

Chunks: ['Employees receive 20 days of paid annual leave per year.', 'Sick leave is granted for up to 10 days per year.', 'Passwords must be changed every 90 days.']
Best chunk for the question: Employees receive 20 days of paid annual leave per year.


In [21]:
%%capture
!pip install "langchain==0.2.7"
!pip install "langchain-core==0.2.20"
!pip install "langchain-text-splitters==0.2.2"
!pip install "lxml==5.2.2"
!pip install tiktoken        # optional, only for the token section

# ---- Sample document used throughout the lesson ----
companypolicies = """Company Policies Handbook

1. Code of Conduct
All employees must act with honesty and respect. Harassment of any kind is not tolerated.
Employees should report any violation to the Human Resources department immediately.

2. Remote Work Policy
Employees may work remotely up to three days per week with manager approval.
Remote workers must be available during core hours, from 10 AM to 3 PM.
Company equipment must be kept secure at all times.

3. Leave Policy
Full-time employees receive 20 days of paid annual leave per year.
Sick leave is granted for up to 10 days per year and may require a doctor's note.
Parental leave is available to all employees after six months of service.

4. Information Security
Passwords must be at least 12 characters long and changed every 90 days.
Confidential data must never be shared through personal email accounts.
Lost or stolen devices must be reported to the IT team within 24 hours.
"""

print(len(companypolicies), "characters")   # Output: 927 characters

# ---- Optional: load your own real file instead ----
# with open("companypolicies.txt", "r", encoding="utf-8") as f:
#     companypolicies = f.read()

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# All four dials in one splitter
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],  # Dotted Line: try these cuts, in order
    chunk_size=100,                      # Slice Size: max 100 units per chunk
    chunk_overlap=20,                    # Shared Cheese: 20 units shared
    length_function=len,                 # Ruler Type: count characters
)
chunks = splitter.split_text(companypolicies)
print("Chunks created:", len(chunks))
print("Longest chunk :", max(len(c) for c in chunks), "characters")

# Output:
# Chunks created: 13
# Longest chunk : 98 characters   (never above chunk_size)

# What happens when Shared Cheese is bigger than Slice Size?
try:
    RecursiveCharacterTextSplitter(chunk_size=20, chunk_overlap=50)
except ValueError as error:
    print("Error:", error)

# Output:
# Error: Got a larger chunk overlap (50) than chunk size (20), should be smaller.

Chunks created: 13
Longest chunk : 98 characters
Error: Got a larger chunk overlap (50) than chunk size (20), should be smaller.


In [23]:
from langchain_core.documents import Document

doc = Document(
    page_content="""Python is an interpreted high-level general-purpose programming language.
                    Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",
    metadata={
        "my_document_id": 234234,
        "my_document_source": "About Python",
        "my_document_create_time": 1680013019,
    },
)

print(doc.page_content)                    # the text inside the box
print(doc.metadata)                        # the whole label
print(doc.metadata["my_document_source"])  # one item from the label

# Output (last line): About Python

Python is an interpreted high-level general-purpose programming language.
                    Python's design philosophy emphasizes code readability with its notable use of significant indentation.
{'my_document_id': 234234, 'my_document_source': 'About Python', 'my_document_create_time': 1680013019}
About Python


In [24]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=10)

# Button 1: strings in, strings out
strings = splitter.split_text(companypolicies)
print(type(strings[0]))             # <class 'str'>

# Button 2: strings in, Documents out (you add the label)
docs = splitter.create_documents(
    [companypolicies],
    metadatas=[{"source": "Company Policies"}],
)
print(type(docs[0]))                # Document
print(docs[0].metadata)             # {'source': 'Company Policies'}

# Button 3: Documents in, Documents out (labels are copied to every chunk)
original = [Document(page_content=companypolicies, metadata={"source": "Company Policies"})]
chunked = splitter.split_documents(original)
print(len(chunked), chunked[0].metadata)

<class 'str'>
<class 'langchain_core.documents.base.Document'>
{'source': 'Company Policies'}
26 {'source': 'Company Policies'}


In [25]:
from langchain.text_splitter import CharacterTextSplitter

# ---------- Part 1: the blind knife on our policies ----------
text_splitter = CharacterTextSplitter(
    separator="",         # cut anywhere
    chunk_size=200,       # max 200 characters per chunk
    chunk_overlap=20,     # share 20 characters between neighbours
    length_function=len,  # measure length in characters
)
texts = text_splitter.split_text(companypolicies)
print("Number of chunks:", len(texts))            # Output: 6

# Same split, but as Document objects that carry metadata
docs = text_splitter.create_documents(
    [companypolicies],
    metadatas=[{"document": "Company Policies"}],
)
print(docs[0].metadata)                           # {'document': 'Company Policies'}

# ---------- Part 2: a tiny text you can check by eye ----------
story = ("The cat sat on the mat. It was a sunny day. The cat was very happy. "
         "Then a dog came. The dog barked loudly. The cat ran away.")

no_overlap = CharacterTextSplitter(separator="", chunk_size=40, chunk_overlap=0)
for chunk in no_overlap.split_text(story):
    print(repr(chunk))
# Output:
# 'The cat sat on the mat. It was a sunny d'
# 'ay. The cat was very happy. Then a dog c'
# 'ame. The dog barked loudly. The cat ran'
# 'away.'
# Notice: "day" was cut into 'd' + 'ay', and "came" into 'c' + 'ame'.

# ---------- Part 3: add overlap ----------
with_overlap = CharacterTextSplitter(separator="", chunk_size=40, chunk_overlap=10)
for chunk in with_overlap.split_text(story):
    print(repr(chunk))
# Output:
# 'The cat sat on the mat. It was a sunny d'
# 'a sunny day. The cat was very happy. Th'
# 'happy. Then a dog came. The dog barked'
# 'og barked loudly. The cat ran away.'
# Each chunk now starts with a bit of the previous one, so "day" appears whole.

# ---------- Part 4: real separators on three paragraphs ----------
para = (
    "Paragraph one is about cats.\nCats are small.\n\n"
    "Paragraph two is about dogs.\nDogs are loud.\n\n"
    "Paragraph three is about birds.\nBirds can fly."
)

# Cut on blank lines: one paragraph per chunk
by_paragraph = CharacterTextSplitter(separator="\n\n", chunk_size=50, chunk_overlap=0)
print(by_paragraph.split_text(para))
# Output:
# ['Paragraph one is about cats.\nCats are small.',
#  'Paragraph two is about dogs.\nDogs are loud.',
#  'Paragraph three is about birds.\nBirds can fly.']

# Cut on single new lines with a small chunk size: one line per chunk
by_line = CharacterTextSplitter(separator="\n", chunk_size=35, chunk_overlap=0)
print(len(by_line.split_text(para)), "chunks")    # Output: 6 chunks

# The oversize problem: paragraphs are 43 to 46 characters, but the limit is 30
too_small = CharacterTextSplitter(separator="\n\n", chunk_size=30, chunk_overlap=0)
chunks = too_small.split_text(para)
print([len(c) for c in chunks])
# Output: [44, 43, 46]
# LangChain also prints warnings like:
# "Created a chunk of size 44, which is longer than the specified 30"
# The splitter has only ONE separator, so it cannot cut the paragraphs any smaller.
# This exact problem is solved by the recursive splitter in the next section.

Number of chunks: 6
{'document': 'Company Policies'}
'The cat sat on the mat. It was a sunny d'
'ay. The cat was very happy. Then a dog c'
'ame. The dog barked loudly. The cat ran'
'away.'
'The cat sat on the mat. It was a sunny d'
'a sunny day. The cat was very happy. Th'
'happy. Then a dog came. The dog barked'
'og barked loudly. The cat ran away.'
['Paragraph one is about cats.\nCats are small.', 'Paragraph two is about dogs.\nDogs are loud.', 'Paragraph three is about birds.\nBirds can fly.']
6 chunks
[44, 43, 46]


In [26]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ---------- Part 1: basic usage on our policies ----------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
)
texts = text_splitter.create_documents([companypolicies])
print("Number of chunks:", len(texts))                 # Output: 13
for t in texts[:3]:
    print(repr(t.page_content))
# Output:
# 'Company Policies Handbook'
# '1. Code of Conduct'
# 'All employees must act with honesty and respect. Harassment of any kind is not tolerated.'

# ---------- Part 2: no word is cut in half ----------
story = ("The cat sat on the mat. It was a sunny day. The cat was very happy. "
         "Then a dog came. The dog barked loudly. The cat ran away.")

splitter = RecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=10)
for chunk in splitter.split_text(story):
    print(repr(chunk))
# Output:
# 'The cat sat on the mat. It was a sunny day. The cat was very'
# 'was very happy. Then a dog came. The dog barked loudly. The'
# 'The cat ran away.'
# Every chunk ends on a whole word, because the splitter fell back to spaces.

# ---------- Part 3: the problem from the previous section is gone ----------
para = (
    "Paragraph one is about cats.\nCats are small.\n\n"
    "Paragraph two is about dogs.\nDogs are loud.\n\n"
    "Paragraph three is about birds.\nBirds can fly."
)
small = RecursiveCharacterTextSplitter(chunk_size=30, chunk_overlap=0)
print(small.split_text(para))
# Output:
# ['Paragraph one is about cats.', 'Cats are small.',
#  'Paragraph two is about dogs.', 'Dogs are loud.',
#  'Paragraph three is about', 'birds.', 'Birds can fly.']
# No warnings: the 31-character line was cut on a space, not in the middle of a word.

# ---------- Part 4: your own list of separators ----------
sentence_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],   # added ". " so sentences are preferred
    chunk_size=80,
    chunk_overlap=10,
)
for chunk in sentence_splitter.split_text(story):
    print(repr(chunk))
# Output:
# 'The cat sat on the mat. It was a sunny day. The cat was very happy'
# '. Then a dog came. The dog barked loudly. The cat ran away.'
# Note: by default the separator (". ") is kept at the START of the next chunk,
# which is why the second chunk begins with a full stop. The keep_separator
# parameter controls this behaviour.

Number of chunks: 13
'Company Policies Handbook'
'1. Code of Conduct'
'All employees must act with honesty and respect. Harassment of any kind is not tolerated.'
'The cat sat on the mat. It was a sunny day. The cat was very'
'was very happy. Then a dog came. The dog barked loudly. The'
'The cat ran away.'
['Paragraph one is about cats.', 'Cats are small.', 'Paragraph two is about dogs.', 'Dogs are loud.', 'Paragraph three is about', 'birds.', 'Birds can fly.']
'The cat sat on the mat. It was a sunny day. The cat was very happy'
'. Then a dog came. The dog barked loudly. The cat ran away.'


In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

# ---------- Which languages are supported? ----------
print([lang.value for lang in Language])
# Output (depends on your version): ['cpp', 'go', 'java', 'kotlin', 'js', 'ts', 'php',
# 'proto', 'python', 'r', 'rst', 'ruby', 'rust', 'scala', 'swift', 'markdown', 'latex',
# 'html', 'sol', 'csharp', 'cobol', 'c', 'lua', 'perl', 'haskell', 'elixir', ...]

# ---------- Python ----------
PYTHON_CODE = """
def hello_world():
    print("Hello, World!")

# Call the function
hello_world()
"""

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0
)
for d in python_splitter.create_documents([PYTHON_CODE]):
    print(repr(d.page_content))
# Output:
# 'def hello_world():\n    print("Hello, World!")'
# '# Call the function\nhello_world()'

# Peek at the separators used for Python
print(RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON))
# Output: ['\nclass ', '\ndef ', '\n\tdef ', '\n\n', '\n', ' ', '']
# Read it as: cut before a class, else before a def, else before an indented def,
# else on blank lines, lines, words, letters.

# ---------- JavaScript ----------
print(RecursiveCharacterTextSplitter.get_separators_for_language(Language.JS))
# Output starts with: ['\nfunction ', '\nconst ', '\nlet ', '\nvar ', '\nclass ', ...]

JS_CODE = """
function helloWorld() {
  console.log("Hello, World!");
}

// Call the function
helloWorld();
"""

js_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.JS, chunk_size=60, chunk_overlap=0
)
for d in js_splitter.create_documents([JS_CODE]):
    print(repr(d.page_content))
# Output:
# 'function helloWorld() {\n  console.log("Hello, World!");\n}'
# '// Call the function\nhelloWorld();'

# ---------- Any other language: change only Language.XXX ----------
java_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.JAVA, chunk_size=200, chunk_overlap=20
)

['cpp', 'go', 'java', 'kotlin', 'js', 'ts', 'php', 'proto', 'python', 'rst', 'ruby', 'rust', 'scala', 'swift', 'markdown', 'latex', 'html', 'sol', 'csharp', 'cobol', 'c', 'lua', 'perl', 'haskell', 'elixir']
'def hello_world():\n    print("Hello, World!")'
'# Call the function\nhello_world()'
['\nclass ', '\ndef ', '\n\tdef ', '\n\n', '\n', ' ', '']
['\nfunction ', '\nconst ', '\nlet ', '\nvar ', '\nclass ', '\nif ', '\nfor ', '\nwhile ', '\nswitch ', '\ncase ', '\ndefault ', '\n\n', '\n', ' ', '']
'function helloWorld() {\n  console.log("Hello, World!");\n}'
'// Call the function\nhelloWorld();'


In [28]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

md = """# Pets
Intro to pets.
## Cats
Cats purr.
### Kittens
Kittens are tiny.
## Dogs
Dogs bark."""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

# ---------- Default: headers go to metadata only ----------
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
for d in markdown_splitter.split_text(md):
    print(d)
# Output:
# page_content='Intro to pets.' metadata={'Header 1': 'Pets'}
# page_content='Cats purr.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats'}
# page_content='Kittens are tiny.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats', 'Header 3': 'Kittens'}
# page_content='Dogs bark.' metadata={'Header 1': 'Pets', 'Header 2': 'Dogs'}
# Look at "Kittens are tiny.": its metadata holds the whole address Pets > Cats > Kittens.

# ---------- strip_headers=False: headers stay inside the text too ----------
keep_headers = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,
)
for d in keep_headers.split_text(md):
    print(d)
# Output:
# page_content='# Pets\nIntro to pets.' metadata={'Header 1': 'Pets'}
# page_content='## Cats\nCats purr.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats'}
# page_content='### Kittens\nKittens are tiny.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats', 'Header 3': 'Kittens'}
# page_content='## Dogs\nDogs bark.' metadata={'Header 1': 'Pets', 'Header 2': 'Dogs'}

page_content='Intro to pets.' metadata={'Header 1': 'Pets'}
page_content='Cats purr.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats'}
page_content='Kittens are tiny.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats', 'Header 3': 'Kittens'}
page_content='Dogs bark.' metadata={'Header 1': 'Pets', 'Header 2': 'Dogs'}
page_content='# Pets
Intro to pets.' metadata={'Header 1': 'Pets'}
page_content='## Cats
Cats purr.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats'}
page_content='### Kittens
Kittens are tiny.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats', 'Header 3': 'Kittens'}
page_content='## Dogs
Dogs bark.' metadata={'Header 1': 'Pets', 'Header 2': 'Dogs'}


In [29]:
from langchain_text_splitters import HTMLHeaderTextSplitter, HTMLSectionSplitter

html_string = """<html><body><div><h1>Foo</h1><p>Intro Foo.</p><div><h2>Bar</h2><p>Intro Bar.</p><h3>Sub 1</h3><p>Text 1.</p></div><div><h2>Baz</h2><p>Text Baz</p></div></div></body></html>"""

headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]

# ---------- Robot A: split by HTML header ----------
header_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
for d in header_splitter.split_text(html_string):
    print(d)
# Output:
# page_content='Foo' metadata={'Header 1': 'Foo'}
# page_content='Intro Foo.' metadata={'Header 1': 'Foo'}
# page_content='Bar' metadata={'Header 1': 'Foo', 'Header 2': 'Bar'}
# page_content='Intro Bar.' metadata={'Header 1': 'Foo', 'Header 2': 'Bar'}
# page_content='Sub 1' metadata={'Header 1': 'Foo', 'Header 2': 'Bar', 'Header 3': 'Sub 1'}
# page_content='Text 1.' metadata={'Header 1': 'Foo', 'Header 2': 'Bar', 'Header 3': 'Sub 1'}
# page_content='Baz' metadata={'Header 1': 'Foo', 'Header 2': 'Baz'}
# page_content='Text Baz' metadata={'Header 1': 'Foo', 'Header 2': 'Baz'}

# ---------- Robot B: split by HTML section ----------
section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)
for d in section_splitter.split_text(html_string):
    print(d)
# Output:
# page_content='Foo \n Intro Foo.' metadata={'Header 1': 'Foo'}
# page_content='Bar \n Intro Bar.' metadata={'Header 2': 'Bar'}
# page_content='Sub 1 \n Text 1.' metadata={'Header 3': 'Sub 1'}
# page_content='Baz \n Text Baz' metadata={'Header 2': 'Baz'}

# ---------- Optional: split a live web page (needs internet) ----------
# docs = header_splitter.split_text_from_url("https://example.com")

page_content='Foo'
page_content='Intro Foo.  
BarSub 1' metadata={'Header 1': 'Foo'}
page_content='Intro Bar.' metadata={'Header 1': 'Foo', 'Header 2': 'Bar'}
page_content='Text 1.' metadata={'Header 1': 'Foo', 'Header 2': 'Bar', 'Header 3': 'Sub 1'}
page_content='Baz' metadata={'Header 1': 'Foo'}
page_content='Text Baz' metadata={'Header 1': 'Foo', 'Header 2': 'Baz'}
page_content='Foo 
 Intro Foo.' metadata={'Header 1': 'Foo'}
page_content='Bar 
 Intro Bar.' metadata={'Header 2': 'Bar'}
page_content='Sub 1 
 Text 1.' metadata={'Header 3': 'Sub 1'}
page_content='Baz 
 Text Baz' metadata={'Header 2': 'Baz'}


In [30]:
from langchain_text_splitters import RecursiveJsonSplitter

data = {
    "a": {"b": "x" * 30, "c": "y" * 30},
    "d": [1, 2, 3],
}

json_splitter = RecursiveJsonSplitter(max_chunk_size=60)
for chunk in json_splitter.split_text(data):
    print(chunk)

# Output:
# {"a": {"b": "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx", "c": "yyyyyyyyyyyyyyyyyyyyyyyyyyyyyy"}}
# {"d": [1, 2, 3]}
# Each line is a complete, valid JSON object.

{"a": {"b": "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx", "c": "yyyyyyyyyyyyyyyyyyyyyyyyyyyyyy"}}
{"d": [1, 2, 3]}


In [31]:
from langchain_text_splitters import TokenTextSplitter, RecursiveCharacterTextSplitter

# ---------- Option A: pure token splitting ----------
token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",  # tokenizer used by many OpenAI models
    chunk_size=50,                # 50 TOKENS, not characters
    chunk_overlap=5,
)
token_chunks = token_splitter.split_text(companypolicies)
print("Option A chunks:", len(token_chunks))

# ---------- Option B: recursive separators, measured in tokens ----------
token_recursive = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=50,
    chunk_overlap=5,
)
chunks_b = token_recursive.split_text(companypolicies)
print("Option B chunks:", len(chunks_b))

# ---------- Option C: your own ruler (count words) ----------
def word_count(text: str) -> int:
    return len(text.split())

word_splitter = RecursiveCharacterTextSplitter(
    chunk_size=20,              # 20 WORDS
    chunk_overlap=3,
    length_function=word_count,
)
chunks_c = word_splitter.split_text(companypolicies)
print("Option C chunks:", len(chunks_c))     # Output: 12
print(repr(chunks_c[0]))                     # Output: 'Company Policies Handbook'

Option A chunks: 5
Option B chunks: 6
Option C chunks: 12
'Company Policies Handbook'


In [32]:
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

md = """# Pets
Intro to pets.
## Cats
Cats purr.
### Kittens
Kittens are tiny.
## Dogs
Dogs bark."""

headers_to_split_on = [("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")]

# Stage 1: cut by chapters (structure)
stage1_docs = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
).split_text(md)

# Stage 2: make sure no chunk is bigger than chunk_size (size)
stage2 = RecursiveCharacterTextSplitter(chunk_size=12, chunk_overlap=0)
final_docs = stage2.split_documents(stage1_docs)

for d in final_docs:
    print(d)

# Output (chunk_size=12 is deliberately tiny, just to show the effect):
# page_content='Intro to' metadata={'Header 1': 'Pets'}
# page_content='pets.' metadata={'Header 1': 'Pets'}
# page_content='Cats purr.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats'}
# page_content='Kittens are' metadata={'Header 1': 'Pets', 'Header 2': 'Cats', 'Header 3': 'Kittens'}
# page_content='tiny.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats', 'Header 3': 'Kittens'}
# page_content='Dogs bark.' metadata={'Header 1': 'Pets', 'Header 2': 'Dogs'}
# "Intro to pets." was too long, so it became two pieces, and BOTH kept the label.

page_content='Intro to' metadata={'Header 1': 'Pets'}
page_content='pets.' metadata={'Header 1': 'Pets'}
page_content='Cats purr.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats'}
page_content='Kittens are' metadata={'Header 1': 'Pets', 'Header 2': 'Cats', 'Header 3': 'Kittens'}
page_content='tiny.' metadata={'Header 1': 'Pets', 'Header 2': 'Cats', 'Header 3': 'Kittens'}
page_content='Dogs bark.' metadata={'Header 1': 'Pets', 'Header 2': 'Dogs'}


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Compare several chunk sizes (overlap = 10 percent of the size)
for size in (100, 200, 400, 800):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 10)
    chunks = splitter.split_text(companypolicies)
    sizes = [len(c) for c in chunks]
    print(f"chunk_size={size:4} -> chunks={len(chunks):2} | smallest={min(sizes):3} | largest={max(sizes):3}")

# Output:
# chunk_size= 100 -> chunks=13 | smallest= 18 | largest= 98
# chunk_size= 200 -> chunks= 8 | smallest= 25 | largest=193
# chunk_size= 400 -> chunks= 4 | smallest=220 | largest=240
# chunk_size= 800 -> chunks= 2 | smallest=240 | largest=684
# Reading the table: bigger chunk_size means fewer, bigger chunks.

chunk_size= 100 -> chunks=13 | smallest= 18 | largest= 98
chunk_size= 200 -> chunks= 8 | smallest= 25 | largest=193
chunk_size= 400 -> chunks= 4 | smallest=220 | largest=240
chunk_size= 800 -> chunks= 2 | smallest=240 | largest=684


In [34]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    HTMLHeaderTextSplitter,
    RecursiveJsonSplitter,
    Language,
)

def choose_splitter(file_type: str):
    """Return a sensible default splitter for a given file type."""
    file_type = file_type.lower()

    if file_type in ("md", "markdown"):
        return MarkdownHeaderTextSplitter(
            headers_to_split_on=[("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")],
            strip_headers=False,
        )
    if file_type in ("html", "htm"):
        return HTMLHeaderTextSplitter(
            headers_to_split_on=[("h1", "Header 1"), ("h2", "Header 2"), ("h3", "Header 3")]
        )
    if file_type == "json":
        return RecursiveJsonSplitter(max_chunk_size=500)
    if file_type in ("py", "python"):
        return RecursiveCharacterTextSplitter.from_language(
            language=Language.PYTHON, chunk_size=800, chunk_overlap=0)
    if file_type in ("js", "javascript"):
        return RecursiveCharacterTextSplitter.from_language(
            language=Language.JS, chunk_size=800, chunk_overlap=0)
    if file_type in ("tex", "latex"):
        return RecursiveCharacterTextSplitter.from_language(
            language=Language.LATEX, chunk_size=800, chunk_overlap=0)

    # Safe default for everything else (plain text, articles, ...)
    return RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

for kind in ["txt", "md", "html", "json", "py"]:
    print(kind, "->", type(choose_splitter(kind)).__name__)

# Output:
# txt -> RecursiveCharacterTextSplitter
# md -> MarkdownHeaderTextSplitter
# html -> HTMLHeaderTextSplitter
# json -> RecursiveJsonSplitter
# py -> RecursiveCharacterTextSplitter

txt -> RecursiveCharacterTextSplitter
md -> MarkdownHeaderTextSplitter
html -> HTMLHeaderTextSplitter
json -> RecursiveJsonSplitter
py -> RecursiveCharacterTextSplitter


In [35]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    Language,
)

# ---------- Mistake 1: overlap bigger than chunk size ----------
try:
    CharacterTextSplitter(chunk_size=20, chunk_overlap=50)
except ValueError as error:
    print("Mistake 1:", error)
# Output: Mistake 1: Got a larger chunk overlap (50) than chunk size (20), should be smaller.
# Fix: use chunk_overlap=2 (about 10 percent).

# ---------- Mistake 2: forgetting the raw string for LaTeX ----------
broken = "\begin{document}"        # "\b" becomes a backspace character
fixed = r"\begin{document}"        # raw string keeps the backslash
print("Broken length:", len(broken), "| Fixed length:", len(fixed))
# Output: Broken length: 15 | Fixed length: 16
# The broken string is one character shorter because \b was swallowed.

# ---------- Mistake 3: metadata lost with split_text ----------
splitter = RecursiveCharacterTextSplitter(chunk_size=30, chunk_overlap=0)
lost = splitter.split_text("Alpha beta gamma delta epsilon zeta eta theta")
print(type(lost[0]).__name__)      # Output: str   (no metadata possible!)

kept = splitter.create_documents(
    ["Alpha beta gamma delta epsilon zeta eta theta"],
    metadatas=[{"source": "greek"}],
)
print(kept[0].metadata)            # Output: {'source': 'greek'}

Mistake 1: Got a larger chunk overlap (50) than chunk size (20), should be smaller.
Broken length: 15 | Fixed length: 16
str
{'source': 'greek'}


In [36]:
from langchain_text_splitters import (
    CharacterTextSplitter, RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter, Language,
)

text = "Cats purr.\n\nDogs bark.\n\nBirds sing."

print(CharacterTextSplitter(separator="\n\n", chunk_size=15, chunk_overlap=0).split_text(text))
print(RecursiveCharacterTextSplitter(chunk_size=15, chunk_overlap=0).split_text(text))
print(RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=40, chunk_overlap=0
).split_text("def a():\n    return 1\n\ndef b():\n    return 2"))
print(MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "H1")]).split_text("# Zoo\nLions roar."))

# Output:
# ['Cats purr.', 'Dogs bark.', 'Birds sing.']
# ['Cats purr.', 'Dogs bark.', 'Birds sing.']
# ['def a():\n    return 1', 'def b():\n    return 2']
# [Document(page_content='Lions roar.', metadata={'H1': 'Zoo'})]
# (Exact printing of Document objects differs slightly between versions.)

['Cats purr.', 'Dogs bark.', 'Birds sing.']
['Cats purr.', 'Dogs bark.', 'Birds sing.']
['def a():\n    return 1', 'def b():\n    return 2']
[Document(metadata={'H1': 'Zoo'}, page_content='Lions roar.')]


Solution 1: The Scissors Story


In [37]:
from langchain_text_splitters import CharacterTextSplitter

for sep in ["\n", "\n\n", ""]:
    splitter = CharacterTextSplitter(
        separator=sep,
        chunk_size=200,
        chunk_overlap=20,
        length_function=len,
    )
    docs = splitter.create_documents([companypolicies])
    print(f"separator={sep!r:6} -> {len(docs)} chunks, sizes={[len(d.page_content) for d in docs]}")
    print("   first chunk:", repr(docs[0].page_content[:60]), "...")

# Output (with the sample companypolicies from the Setup section):
# separator='\n'   -> 6 chunks, sizes=[134, 183, 139, 164, 170, 143]
# separator='\n\n' -> 5 chunks, sizes=[25, 193, 222, 238, 240]     (plus warnings: sizes above 200)
# separator=''     -> 6 chunks, sizes=[200, 199, 199, 199, 199, 26]
# Notice: "\n" gives tidy sizes under 200, "\n\n" breaks the limit, and "" is exact but blind.

separator='\n'   -> 6 chunks, sizes=[134, 183, 139, 164, 170, 143]
   first chunk: 'Company Policies Handbook\n1. Code of Conduct\nAll employees m' ...
separator='\n\n' -> 5 chunks, sizes=[25, 193, 222, 238, 240]
   first chunk: 'Company Policies Handbook' ...
separator=''     -> 6 chunks, sizes=[200, 199, 199, 199, 199, 26]
   first chunk: 'Company Policies Handbook\n\n1. Code of Conduct\nAll employees ' ...


Solution 2: The Scientist's Paper


In [38]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

latex_text = r"""
\documentclass{article}

\begin{document}

\maketitle

\section{Introduction}
Large language models (LLMs) are a type of machine learning model that can be trained on vast amounts of text data to generate human-like language. In recent years, LLMs have made significant advances in a variety of natural language processing tasks, including language translation, text generation, and sentiment analysis.

\subsection{History of LLMs}
The earliest LLMs were developed in the 1980s and 1990s, but they were limited by the amount of data that could be processed and the computational power available at the time. In the past decade, however, advances in hardware and software have made it possible to train LLMs on massive datasets, leading to significant improvements in performance.

\subsection{Applications of LLMs}
LLMs have many applications in industry, including chatbots, content creation, and virtual assistants. They can also be used in academia for research in linguistics, psychology, and computational linguistics.

\end{document}
"""

for size in (150, 400):
    latex_splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.LATEX, chunk_size=size, chunk_overlap=0
    )
    latex_docs = latex_splitter.create_documents([latex_text])
    print(f"===== chunk_size={size}: {len(latex_docs)} chunks =====")
    for i, doc in enumerate(latex_docs, start=1):
        first_line = doc.page_content.strip().splitlines()[0][:45]
        print(f"Chunk {i} ({len(doc.page_content)} chars): {first_line}")

# Output:
# ===== chunk_size=150: 9 chunks =====
# Chunk 1 (53 chars): \documentclass{article}
# Chunk 2 (149 chars): \section{Introduction}
# Chunk 3 (143 chars): human-like language. In recent years, LLMs ...
# Chunk 4 (53 chars): translation, text generation, and sentiment analysis.
# Chunk 5 (143 chars): \subsection{History of LLMs}
# Chunk 6 (148 chars): processed and the computational power ...
# Chunk 7 (83 chars): train LLMs on massive datasets, ...
# Chunk 8 (145 chars): \subsection{Applications of LLMs}
# Chunk 9 (112 chars): also be used in academia for research ...
# ===== chunk_size=400: 4 chunks =====
# Chunk 1 (53 chars): \documentclass{article}
# Chunk 2 (347 chars): \section{Introduction}
# Chunk 3 (376 chars): \subsection{History of LLMs}
# Chunk 4 (258 chars): \subsection{Applications of LLMs}
# With 400, every section starts a new chunk and stays in one piece.
# (The first-line previews above are shortened for display; your exact text may differ slightly.)

===== chunk_size=150: 9 chunks =====
Chunk 1 (53 chars): \documentclass{article}
Chunk 2 (149 chars): \section{Introduction}
Chunk 3 (143 chars): human-like language. In recent years, LLMs ha
Chunk 4 (53 chars): translation, text generation, and sentiment a
Chunk 5 (143 chars): \subsection{History of LLMs}
Chunk 6 (148 chars): processed and the computational power availab
Chunk 7 (83 chars): train LLMs on massive datasets, leading to si
Chunk 8 (145 chars): \subsection{Applications of LLMs}
Chunk 9 (112 chars): also be used in academia for research in ling
===== chunk_size=400: 4 chunks =====
Chunk 1 (53 chars): \documentclass{article}
Chunk 2 (347 chars): \section{Introduction}
Chunk 3 (376 chars): \subsection{History of LLMs}
Chunk 4 (258 chars): \subsection{Applications of LLMs}


Solution 3 (Capstone): The Helpful Librarian Robot


In [39]:
import re
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

handbook_md = """# Employee Handbook

## Remote Work
Employees may work remotely up to three days per week with manager approval. Remote workers must be available during core hours, from 10 AM to 3 PM. Company equipment must be kept secure at all times.

## Leave Policy
### Annual Leave
Full-time employees receive 20 days of paid annual leave per year. Leave requests should be submitted at least two weeks in advance.
### Sick Leave
Sick leave is granted for up to 10 days per year and may require a doctor's note.

## Information Security
Passwords must be at least 12 characters long and changed every 90 days. Lost or stolen devices must be reported to the IT team within 24 hours.
"""

# ---------- Task 1: two-stage split ----------
headers_to_split_on = [("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")]

stage1 = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,            # keep the header text inside page_content
).split_text(handbook_md)

stage2 = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
docs = stage2.split_documents(stage1)

# ---------- Task 2: inspect the chunks ----------
for i, d in enumerate(docs, start=1):
    print(f"Chunk {i} | {len(d.page_content)} chars | {d.metadata}")
    print("   ", repr(d.page_content[:60]))

# ---------- Task 3: a tiny retriever ----------
STOPWORDS = {"the", "a", "an", "of", "do", "i", "how", "many", "must", "is", "to", "get", "my", "often"}

def tokenize(text):
    words = re.findall(r"[a-z0-9']+", text.lower())
    return {w[:5] for w in words if w not in STOPWORDS}     # first 5 letters = crude stemming

def retrieve(question, docs, k=2):
    q_words = tokenize(question)
    scored = [(len(q_words & tokenize(d.page_content)), d) for d in docs]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:k]

# ---------- Task 4: test ----------
for q in ["How many days of sick leave do I get?",
          "How often must I change my password?"]:
    print("\nQUESTION:", q)
    for score, d in retrieve(q, docs, k=2):
        print(f"  score={score} | {d.metadata.get('Header 3') or d.metadata.get('Header 2')}")

# Output (Task 2):
# Chunk 1 | 36 chars  | {'Header 1': 'Employee Handbook', 'Header 2': 'Remote Work'}
#     '# Employee Handbook  \n## Remote Work'
# Chunk 2 | 193 chars | {'Header 1': 'Employee Handbook', 'Header 2': 'Remote Work'}
#     'Employees may work remotely up to three days per week with m'
# Chunk 3 | 33 chars  | {'Header 1': 'Employee Handbook', 'Header 2': 'Remote Work'}
#     'must be kept secure at all times.'
# Chunk 4 | 167 chars | {... 'Header 2': 'Leave Policy', 'Header 3': 'Annual Leave'}
#     '## Leave Policy  \n### Annual Leave\nFull-time employees recei'
# Chunk 5 | 96 chars  | {... 'Header 2': 'Leave Policy', 'Header 3': 'Sick Leave'}
#     '### Sick Leave\nSick leave is granted for up to 10 days per y'
# Chunk 6 | 168 chars | {'Header 1': 'Employee Handbook', 'Header 2': 'Information Security'}
#     '## Information Security\nPasswords must be at least 12 charac'
#
# Output (Task 4):
# QUESTION: How many days of sick leave do I get?
#   score=3 | Sick Leave
#   score=2 | Annual Leave
#
# QUESTION: How often must I change my password?
#   score=2 | Information Security
#   score=0 | Remote Work

Chunk 1 | 36 chars | {'Header 1': 'Employee Handbook', 'Header 2': 'Remote Work'}
    '# Employee Handbook  \n## Remote Work'
Chunk 2 | 193 chars | {'Header 1': 'Employee Handbook', 'Header 2': 'Remote Work'}
    'Employees may work remotely up to three days per week with m'
Chunk 3 | 33 chars | {'Header 1': 'Employee Handbook', 'Header 2': 'Remote Work'}
    'must be kept secure at all times.'
Chunk 4 | 167 chars | {'Header 1': 'Employee Handbook', 'Header 2': 'Leave Policy', 'Header 3': 'Annual Leave'}
    '## Leave Policy  \n### Annual Leave\nFull-time employees recei'
Chunk 5 | 96 chars | {'Header 1': 'Employee Handbook', 'Header 2': 'Leave Policy', 'Header 3': 'Sick Leave'}
    '### Sick Leave\nSick leave is granted for up to 10 days per y'
Chunk 6 | 168 chars | {'Header 1': 'Employee Handbook', 'Header 2': 'Information Security'}
    '## Information Security\nPasswords must be at least 12 charac'

QUESTION: How many days of sick leave do I get?
  score=3 | Sick Leave
  score=2 | 